# Stage 1 — Environment Setup
## Project: Optimizing Indonesian Sentence Embeddings for STS
### A Contrastive Fine-Tuning Study with Hard Negative Mining

---
**Pipeline Stage:** 1 of 12  
**Notebook:** `01_environment_setup.ipynb`  
**Objective:** Install semua dependencies, mount Google Drive, buat struktur folder, dan verifikasi environment siap digunakan.

---

## 1.1 — Install Library Dependencies

Install semua package yang dibutuhkan sepanjang proyek ini.

> ⚠️ **Tips:** Setelah install, jangan restart runtime kecuali diminta. Jika Colab meminta restart, klik **Restart** lalu jalankan ulang cell berikutnya.

In [1]:
# ============================================================
# CELL 1.1 — Install Core Dependencies
# ============================================================
# Runtime: ~3-5 menit pada Colab

print("[INFO] Installing core dependencies...")

# sentence-transformers: framework utama untuk sentence embeddings
# Versi di-pin agar konsisten dengan blueprint
!pip install -q sentence-transformers==2.7.0

# transformers + datasets: HuggingFace ecosystem
!pip install -q transformers==4.40.0 datasets==2.19.0

# rank_bm25: untuk Hard Negative Mining (Stage 8)
!pip install -q rank_bm25

# huggingface_hub: untuk upload model ke HF Hub (Stage 12)
!pip install -q huggingface_hub

# gradio: untuk Gradio demo (Stage 12)
!pip install -q gradio

# scipy, sklearn, matplotlib: evaluasi dan visualisasi
!pip install -q scipy scikit-learn matplotlib seaborn

# accelerate: optimasi training PyTorch di Colab
!pip install -q accelerate

print("[OK] Semua dependencies berhasil diinstall.")

[INFO] Installing core dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3

## 1.2 — Mount Google Drive

Semua data, model, dan hasil eksperimen disimpan di Google Drive agar tidak hilang saat Colab session berakhir.

> ⚠️ **Tips:** Klik popup autentikasi yang muncul, pilih akun Google, dan izinkan akses.

In [2]:
# ============================================================
# CELL 1.2 — Mount Google Drive
# ============================================================

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# Verifikasi mount berhasil
if os.path.exists('/content/drive/MyDrive'):
    print("[OK] Google Drive berhasil di-mount di /content/drive/MyDrive")
else:
    raise RuntimeError("[ERROR] Google Drive gagal di-mount. Coba jalankan ulang cell ini.")

Mounted at /content/drive
[OK] Google Drive berhasil di-mount di /content/drive/MyDrive


## 1.3 — Konfigurasi Path (Project Config)

Semua path project didefinisikan di satu tempat agar mudah di-maintain.
Konfigurasi ini akan di-import oleh semua notebook lain dalam proyek.

In [3]:
# ============================================================
# CELL 1.3 — Project Configuration (Source of Truth untuk semua paths)
# ============================================================

import os
from pathlib import Path

# ── Root Project ──────────────────────────────────────────────
GDRIVE_ROOT  = Path("/content/drive/MyDrive")
PROJECT_ROOT = GDRIVE_ROOT / "AI-Projects" / "sts-indonesian-embeddings"

# ── Sub-directories ──────────────────────────────────────────
PATHS = {
    # Dataset storage
    "datasets"        : PROJECT_ROOT / "datasets",
    "indosts"         : PROJECT_ROOT / "datasets" / "indosts",
    "stsb_id"         : PROJECT_ROOT / "datasets" / "stsb-id",
    "mining_corpus"   : PROJECT_ROOT / "datasets" / "mining-corpus",

    # Data splits (processed)
    "splits"          : PROJECT_ROOT / "datasets" / "splits",

    # Experiments
    "experiments"     : PROJECT_ROOT / "experiments",
    "baseline_exp"    : PROJECT_ROOT / "experiments" / "baseline",
    "simcse_exp"      : PROJECT_ROOT / "experiments" / "simcse",
    "sbert_exp"       : PROJECT_ROOT / "experiments" / "sbert-hard-negatives",

    # Saved model checkpoints
    "models"          : PROJECT_ROOT / "models",
    "baseline_model"  : PROJECT_ROOT / "models" / "baseline",
    "simcse_model"    : PROJECT_ROOT / "models" / "simcse",
    "sbert_model"     : PROJECT_ROOT / "models" / "sbert",

    # HuggingFace cache (agar tidak re-download)
    "hf_cache"        : PROJECT_ROOT / "hf_cache",

    # Logs, embeddings, evaluation outputs
    "logs"            : PROJECT_ROOT / "logs",
    "embeddings"      : PROJECT_ROOT / "embeddings",
    "evaluation"      : PROJECT_ROOT / "evaluation",

    # Gradio demo assets
    "demo"            : PROJECT_ROOT / "demo",
}

# Buat semua direktori (idempotent)
for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)

print("[OK] Struktur folder project berhasil dibuat:")
print(f"     Root: {PROJECT_ROOT}")
print()

# Tampilkan tree struktur
for name, path in PATHS.items():
    rel = path.relative_to(GDRIVE_ROOT)
    print(f"  ✓  MyDrive/{rel}")

[OK] Struktur folder project berhasil dibuat:
     Root: /content/drive/MyDrive/AI-Projects/sts-indonesian-embeddings

  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/datasets
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/datasets/indosts
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/datasets/stsb-id
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/datasets/mining-corpus
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/datasets/splits
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/experiments
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/experiments/baseline
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/experiments/simcse
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/experiments/sbert-hard-negatives
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/models
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/models/baseline
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/models/simcse
  ✓  MyDrive/AI-Projects/sts-indonesian-embeddings/models/sbert
 

## 1.4 — Simpan Config ke File (untuk di-import notebook lain)

Kita simpan semua path config ke file `project_config.py` di Google Drive agar bisa di-import dari notebook mana pun.

In [4]:
# ============================================================
# CELL 1.4 — Generate project_config.py
# ============================================================

CONFIG_PY = PROJECT_ROOT / "project_config.py"

config_content = '''
"""
project_config.py
─────────────────
Centralized path configuration untuk seluruh notebook.
Cara penggunaan di notebook lain:

    import sys
    sys.path.insert(0, "/content/drive/MyDrive/AI-Projects/sts-indonesian-embeddings")
    from project_config import PATHS, PROJECT_ROOT, HF_CACHE_DIR
"""

from pathlib import Path

GDRIVE_ROOT   = Path("/content/drive/MyDrive")
PROJECT_ROOT  = GDRIVE_ROOT / "AI-Projects" / "sts-indonesian-embeddings"
HF_CACHE_DIR  = str(PROJECT_ROOT / "hf_cache")

PATHS = {
    "datasets"        : PROJECT_ROOT / "datasets",
    "indosts"         : PROJECT_ROOT / "datasets" / "indosts",
    "stsb_id"         : PROJECT_ROOT / "datasets" / "stsb-id",
    "mining_corpus"   : PROJECT_ROOT / "datasets" / "mining-corpus",
    "splits"          : PROJECT_ROOT / "datasets" / "splits",
    "experiments"     : PROJECT_ROOT / "experiments",
    "baseline_exp"    : PROJECT_ROOT / "experiments" / "baseline",
    "simcse_exp"      : PROJECT_ROOT / "experiments" / "simcse",
    "sbert_exp"       : PROJECT_ROOT / "experiments" / "sbert-hard-negatives",
    "models"          : PROJECT_ROOT / "models",
    "baseline_model"  : PROJECT_ROOT / "models" / "baseline",
    "simcse_model"    : PROJECT_ROOT / "models" / "simcse",
    "sbert_model"     : PROJECT_ROOT / "models" / "sbert",
    "hf_cache"        : PROJECT_ROOT / "hf_cache",
    "logs"            : PROJECT_ROOT / "logs",
    "embeddings"      : PROJECT_ROOT / "embeddings",
    "evaluation"      : PROJECT_ROOT / "evaluation",
    "demo"            : PROJECT_ROOT / "demo",
}

# Pastikan semua dir ada saat di-import
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)
'''

with open(CONFIG_PY, 'w') as f:
    f.write(config_content)

print(f"[OK] project_config.py tersimpan di:\n     {CONFIG_PY}")

# Test import config
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from project_config import PATHS as _test_paths, PROJECT_ROOT as _pr
print(f"[OK] project_config.py berhasil di-import. ROOT: {_pr}")

[OK] project_config.py tersimpan di:
     /content/drive/MyDrive/AI-Projects/sts-indonesian-embeddings/project_config.py
[OK] project_config.py berhasil di-import. ROOT: /content/drive/MyDrive/AI-Projects/sts-indonesian-embeddings


## 1.5 — Verifikasi GPU

Pastikan Colab menggunakan GPU (T4). Jika tidak ada GPU, training akan sangat lambat.

> ⚠️ **Jika GPU tidak terdeteksi:** Pergi ke `Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save`

In [5]:
# ============================================================
# CELL 1.5 — GPU Verification
# ============================================================

import torch
import subprocess

print("=" * 55)
print("  ENVIRONMENT VERIFICATION")
print("=" * 55)

# --- PyTorch & CUDA ---
print(f"\n[PyTorch]")
print(f"  Version      : {torch.__version__}")
print(f"  CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU Name     : {gpu_name}")
    print(f"  VRAM         : {gpu_mem:.1f} GB")
    print(f"  CUDA Version : {torch.version.cuda}")
    print(f"  Device       : cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("  ⚠️  WARNING: GPU tidak tersedia! Gunakan CPU (training akan lambat).")
    print("  → Solusi: Runtime → Change runtime type → T4 GPU")

# --- nvidia-smi untuk info lebih detail ---
print("\n[nvidia-smi]")
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True
    )
    for line in result.stdout.strip().split("\n"):
        print(f"  {line}")
except FileNotFoundError:
    print("  nvidia-smi tidak tersedia (CPU mode)")

print(f"\n[Device aktif] {DEVICE}")
print("=" * 55)

  ENVIRONMENT VERIFICATION

[PyTorch]
  Version      : 2.10.0+cu128
  CUDA Available: True
  GPU Name     : Tesla T4
  VRAM         : 15.6 GB
  CUDA Version : 12.8
  Device       : cuda:0

[nvidia-smi]
  Tesla T4, 15360 MiB, 580.82.07

[Device aktif] cuda


## 1.6 — Setup HuggingFace Cache & Environment Variables

In [6]:
# ============================================================
# CELL 1.6 — Setup HuggingFace Cache
# ============================================================
# Redirect HF cache ke Google Drive agar model tidak perlu
# re-download setiap kali session baru dibuka.

import os
from project_config import HF_CACHE_DIR

os.environ["HF_HOME"]              = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"]   = HF_CACHE_DIR
os.environ["HF_DATASETS_CACHE"]    = HF_CACHE_DIR
os.environ["SENTENCE_TRANSFORMERS_HOME"] = HF_CACHE_DIR

# Nonaktifkan tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("[OK] HuggingFace cache diarahkan ke:")
print(f"     {HF_CACHE_DIR}")

# Tampilkan ukuran cache jika sudah ada
import shutil
cache_path = HF_CACHE_DIR
if os.path.exists(cache_path):
    total, used, free = shutil.disk_usage(cache_path)
    # Hitung ukuran folder cache saja
    cache_size = sum(
        os.path.getsize(os.path.join(dirpath, f))
        for dirpath, _, files in os.walk(cache_path)
        for f in files
    ) / 1e6
    print(f"     Cache size saat ini: {cache_size:.1f} MB")

[OK] HuggingFace cache diarahkan ke:
     /content/drive/MyDrive/AI-Projects/sts-indonesian-embeddings/hf_cache
     Cache size saat ini: 0.0 MB


## 1.7 — Testing Environment (End-to-End Smoke Test)

Lakukan smoke test dengan cara:
1. Load model `IndoBERT` dari HuggingFace
2. Encode dua kalimat bahasa Indonesia
3. Hitung cosine similarity

Jika berhasil → environment siap 100% untuk Stage 2.

In [7]:
# ============================================================
# CELL 1.7 — Smoke Test: Load IndoBERT + Encode Sentences
# ============================================================
# Estimasi waktu: ~2-4 menit (download IndoBERT ~440 MB)
# Model akan ter-cache di Drive, tidak perlu download ulang.

import os
os.environ["HF_HOME"] = HF_CACHE_DIR  # Pastikan cache aktif

from sentence_transformers import SentenceTransformer, models, util
import torch

print("[INFO] Loading IndoBERT sebagai SentenceTransformer...")
print("       (Download ~440 MB jika belum ada di cache)\n")

# -- Bangun SentenceTransformer dari IndoBERT --
word_emb = models.Transformer(
    "indobenchmark/indobert-base-p1",
    max_seq_length=128,
    do_lower_case=False   # IndoBERT case-sensitive
)
pooling = models.Pooling(
    word_emb.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)
model = SentenceTransformer(modules=[word_emb, pooling], device=str(DEVICE))

print(f"[OK] Model berhasil dimuat.")
print(f"     Embedding dimension : {word_emb.get_word_embedding_dimension()}")
print(f"     Device              : {DEVICE}")
print(f"     Max seq length      : 128 tokens")

# -- Encode pasangan kalimat --
test_pairs = [
    ("Saya suka makan nasi goreng.",   "Nasi goreng adalah makanan favoritku."),
    ("Hari ini cuaca sangat cerah.",   "Presiden sedang berpidato di istana."),
    ("Kucing itu tidur di bawah meja.","Hewan peliharaan itu beristirahat."),
]

print("\n[INFO] Encoding test sentences...")
sentences_a = [p[0] for p in test_pairs]
sentences_b = [p[1] for p in test_pairs]

emb_a = model.encode(sentences_a, normalize_embeddings=True, show_progress_bar=False)
emb_b = model.encode(sentences_b, normalize_embeddings=True, show_progress_bar=False)

print(f"[OK] Embedding shape: {emb_a.shape}  (n_sentences x hidden_dim)")

# -- Hitung cosine similarity --
cosine_scores = util.cos_sim(emb_a, emb_b).diagonal()

print("\n" + "=" * 60)
print("  SMOKE TEST — Cosine Similarity Results (Zero-Shot Baseline)")
print("=" * 60)
for i, (a, b) in enumerate(test_pairs):
    score = cosine_scores[i].item()
    label = "✓ similar" if score > 0.7 else ("~ neutral" if score > 0.4 else "✗ dissimilar")
    print(f"\n  Pair {i+1}:")
    print(f"    A: {a}")
    print(f"    B: {b}")
    print(f"    Cosine Similarity: {score:.4f}  [{label}]")

print("\n" + "=" * 60)
print("  ✅ ENVIRONMENT SIAP — Stage 1 Complete")
print("=" * 60)

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


[INFO] Loading IndoBERT sebagai SentenceTransformer...
       (Download ~440 MB jika belum ada di cache)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[OK] Model berhasil dimuat.
     Embedding dimension : 768
     Device              : cuda
     Max seq length      : 128 tokens

[INFO] Encoding test sentences...
[OK] Embedding shape: (3, 768)  (n_sentences x hidden_dim)

  SMOKE TEST — Cosine Similarity Results (Zero-Shot Baseline)

  Pair 1:
    A: Saya suka makan nasi goreng.
    B: Nasi goreng adalah makanan favoritku.
    Cosine Similarity: 0.8032  [✓ similar]

  Pair 2:
    A: Hari ini cuaca sangat cerah.
    B: Presiden sedang berpidato di istana.
    Cosine Similarity: 0.5448  [~ neutral]

  Pair 3:
    A: Kucing itu tidur di bawah meja.
    B: Hewan peliharaan itu beristirahat.
    Cosine Similarity: 0.7600  [✓ similar]

  ✅ ENVIRONMENT SIAP — Stage 1 Complete


## 1.8 — Verifikasi Akhir: Library Versions

Cetak versi semua library yang digunakan untuk reproducibility.

In [8]:
# ============================================================
# CELL 1.8 — Library Version Check (Reproducibility)
# ============================================================

import torch
import transformers
import sentence_transformers
import datasets
import numpy as np
import scipy
import sklearn

print("=" * 45)
print("  LIBRARY VERSIONS")
print("=" * 45)
versions = {
    "Python"              : __import__('sys').version.split()[0],
    "PyTorch"             : torch.__version__,
    "Transformers"        : transformers.__version__,
    "sentence-transformers": sentence_transformers.__version__,
    "datasets"            : datasets.__version__,
    "numpy"               : np.__version__,
    "scipy"               : scipy.__version__,
    "scikit-learn"        : sklearn.__version__,
}
for lib, ver in versions.items():
    print(f"  {lib:<28}: {ver}")
print("=" * 45)

print("\n✅ Stage 1 selesai! Lanjutkan ke Stage 2 — Dataset Acquisition.")

  LIBRARY VERSIONS
  Python                      : 3.12.12
  PyTorch                     : 2.10.0+cu128
  Transformers                : 4.40.0
  sentence-transformers       : 2.7.0
  datasets                    : 2.19.0
  numpy                       : 2.0.2
  scipy                       : 1.16.3
  scikit-learn                : 1.6.1

✅ Stage 1 selesai! Lanjutkan ke Stage 2 — Dataset Acquisition.


---

## ✅ Stage 1 Complete — Checklist

| Item | Status |
|------|--------|
| Library dependencies installed | ✅ |
| Google Drive mounted | ✅ |
| Project folder structure created | ✅ |
| `project_config.py` saved ke Drive | ✅ |
| HuggingFace cache diarahkan ke Drive | ✅ |
| GPU terdeteksi & terverifikasi | ✅ |
| IndoBERT berhasil diload | ✅ |
| Smoke test cosine similarity OK | ✅ |

---

**Next:** `02_dataset_setup.ipynb` — Dataset Acquisition & Cleaning

---

### ⏱ Estimasi Waktu Stage 1
| Sub-task | Estimasi |
|----------|----------|
| Install libraries | ~3–5 menit |
| Mount Drive + folder setup | ~1 menit |
| Download IndoBERT (pertama kali) | ~2–4 menit |
| Smoke test encoding | ~30 detik |
| **Total** | **~7–10 menit** |